# 🚀 AnimeKu Studio — Gigabit Video Ingestion Engine (Google Colab + Bunny Stream)
Notebook ini menghubungkan daya komputasi **Google Cloud Gigabit Fiber (30–80 MB/s)** ke studio AnimeKu Anda.

### Fitur Utama:
1. **Kecepatan Download Maksimal (30–80 MB/s)** via Google Datacenter & aria2c P2P.
2. **Direct Video Ingestion ke Bunny Stream** (Video Library ID: `824044`).
3. **Terkoneksi Otomatis ke VPS IDCloudHost** (`http://103.52.115.44`) & Database PostgreSQL.
4. **Anti-Sleep Heartbeat** agar sesi Colab tetap standby mendengarkan antrean Admin Panel.

In [ ]:
#@title 🛠️ 1. Persiapan Tools & Repositori (Wajib Dijalankan Pertama Kali) { display-mode: "form" }
import os, subprocess, sys

print("📦 Menginstal aria2c, ffmpeg, ffprobe, dan git...")
subprocess.run("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg git curl", shell=True)

WORKER_DIR = "/content/animeku-worker"
if not os.path.exists(WORKER_DIR):
    print("📥 Melakukan clone repositori animeku-worker dari GitHub...")
    subprocess.run(f"git clone https://github.com/MrHyzee/animeku-worker.git {WORKER_DIR}", shell=True)
else:
    print("🔄 Memperbarui repositori ke commit terbaru...")
    os.chdir(WORKER_DIR)
    subprocess.run("git fetch origin main --quiet && git reset --hard origin/main --quiet", shell=True)

os.chdir(WORKER_DIR)
print("📦 Memeriksa dependencies Node.js...")
subprocess.run("npm install --silent", shell=True)

print("\n✅ [READY] Lingkungan Colab Siap Digunakan!")
print("👉 Lanjut klik tombol Run pada Cell 2 (Konfigurasi) lalu Cell 3 (Jalankan Daemon).")

In [ ]:
#@title 🔑 2. Konfigurasi Kredensial Bunny Stream & VPS IDCloudHost (1-Click Ready) { display-mode: "form" }
import os

BUNNY_LIBRARY_ID = "824044" #@param {type:"string"}
BUNNY_API_KEY = "9c54f545-977e-4a9b-ac2810ab3206-1352-4634" #@param {type:"string"}
BUNNY_CDN_HOSTNAME = "vz-a43d21ad-647.b-cdn.net" #@param {type:"string"}
WORKER_BASE_URL = "http://103.52.115.44" #@param {type:"string"}
STORAGE_DRIVER = "bunny" #@param ["bunny", "r2", "hetzner"]

os.environ["BUNNY_LIBRARY_ID"] = BUNNY_LIBRARY_ID
os.environ["BUNNY_API_KEY"] = BUNNY_API_KEY
os.environ["BUNNY_CDN_HOSTNAME"] = BUNNY_CDN_HOSTNAME
os.environ["WORKER_BASE_URL"] = WORKER_BASE_URL
os.environ["STORAGE_DRIVER"] = STORAGE_DRIVER
os.environ["DEFAULT_MODE"] = "direct"

print("✅ Kredensial Bunny Stream & VPS IDCloudHost Berhasil Diaktifkan!")
print(f"👉 Target Video Library : {BUNNY_LIBRARY_ID}")
print(f"👉 Target CDN Hostname  : {BUNNY_CDN_HOSTNAME}")
print(f"👉 Target Backend VPS   : {WORKER_BASE_URL}")
print(f"👉 Storage Driver       : {STORAGE_DRIVER.upper()}")
print("👉 Siap mendengarkan antrean job Single-File Video berkecepatan 30-80 MB/s.")

In [ ]:
#@title 🚀 3. Jalankan Always-On Worker Daemon (Otomatis Memproses Antrean Admin Panel) { display-mode: "form" }
#@markdown Klik tombol ▶️ di bawah. Server Colab akan standby dan otomatis mengambil job anime apapun yang Anda submit dari Admin Panel.
import os, subprocess, sys
from IPython.display import Javascript, display

WORKER_DIR = "/content/animeku-worker"
if os.path.exists(WORKER_DIR):
    os.chdir(WORKER_DIR)

# Fallback environment otomatis jika Cell 2 dilewati
os.environ["BUNNY_LIBRARY_ID"] = os.environ.get("BUNNY_LIBRARY_ID", "824044")
os.environ["BUNNY_API_KEY"] = os.environ.get("BUNNY_API_KEY", "9c54f545-977e-4a9b-ac2810ab3206-1352-4634")
os.environ["BUNNY_CDN_HOSTNAME"] = os.environ.get("BUNNY_CDN_HOSTNAME", "vz-a43d21ad-647.b-cdn.net")
os.environ["WORKER_BASE_URL"] = os.environ.get("WORKER_BASE_URL", "http://103.52.115.44")
os.environ["STORAGE_DRIVER"] = os.environ.get("STORAGE_DRIVER", "bunny")
os.environ["DEFAULT_MODE"] = "direct"

# 1. Anti-Disconnect Heartbeat: Mensimulasikan heartbeat DOM agar Google Colab tidak tidur/idle
display(Javascript("""
  function ColabHeartbeat() {
    console.log("[AnimeKu Daemon] Heartbeat Keep-Alive: " + new Date().toLocaleTimeString());
    var btn = document.querySelector("colab-connect-button");
    if (btn) btn.click();
  }
  if (!window.animekuHeartbeatInterval) {
    window.animekuHeartbeatInterval = setInterval(ColabHeartbeat, 60000);
    console.log("🛡️ Anti-Sleep Heartbeat Aktif (Interval 60s)!");
  }
"""))

print("🟢 [COLAB GIGABIT DAEMON] Server Colab Aktif Standby Mendengarkan Admin Panel!")
print(f"   👉 Endpoint API VPS  : {os.environ['WORKER_BASE_URL']}")
print(f"   👉 Storage Output    : Bunny Stream Library #{os.environ['BUNNY_LIBRARY_ID']}\n")
print("🔄 Memperbarui script runner ke commit terbaru dari GitHub...")
subprocess.run("git fetch origin main --quiet && git reset --hard origin/main --quiet", shell=True)

cmd = "node scripts/ingest.js worker-daemon --mode direct --worker google-colab"
print(f"⚡ Menjalankan Daemon: {cmd}\n")

# 2. Live stream output seketika ke terminal Colab (Zero-Buffering)
proc = subprocess.Popen(cmd, shell=True)
proc.wait()

if proc.returncode == 0:
    print("\n👋 Daemon dihentikan secara normal.")
else:
    print(f"\n❌ Proses terhenti dengan error code {proc.returncode}.")

In [ ]:
#@title ⚡ 4. Manual Ingest 1 Episode Tertentu (Opsional Langsung dari Sini) { display-mode: "form" }
ANIME_SLUG = "solo-leveling" #@param {type:"string"}
EPISODE_NUMBER = "1" #@param {type:"string"}
MAGNET_URL = "" #@param {type:"string"}

import os, subprocess
WORKER_DIR = "/content/animeku-worker"
if os.path.exists(WORKER_DIR):
    os.chdir(WORKER_DIR)

cmd = f"node scripts/ingest.js auto-ingest --anime '{ANIME_SLUG}' --ep '{EPISODE_NUMBER}' --mode direct"
if MAGNET_URL.strip():
    cmd += f" --magnet '{MAGNET_URL.strip()}'"

print(f"🚀 Menjalankan Ingest Manual Gigabit: {cmd}\n")
subprocess.run(cmd, shell=True)